# Разделитель песен и восстановление аудио — T4

Этот блокнот запускает экспериментальную ветку с топовыми RoFormer-моделями, DeReverb и доступным микшером стемов.

1. Выбери **Среда выполнения → Сменить среду выполнения → T4 GPU**.
2. Нажми **Выполнить всё**.
3. В последней ячейке открой временную ссылку Gradio.
4. Модели скачиваются лениво при первом выборе и повторно не загружаются, пока жив процесс и хватает памяти.


In [ ]:
import shutil
import subprocess

if not shutil.which("nvidia-smi"):
    raise RuntimeError(
        "GPU не найдена. В Colab выбери: Среда выполнения → "
        "Сменить среду выполнения → T4 GPU."
    )

subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(
    [
        "apt-get", "install", "-y", "-qq",
        "ffmpeg", "git-lfs", "espeak-ng",
    ],
    check=True,
)
subprocess.run(["git", "lfs", "install"], check=True)

if not shutil.which("uv") and not shutil.which("/root/.local/bin/uv"):
    subprocess.run(
        ["bash", "-lc", "curl -LsSf https://astral.sh/uv/install.sh | sh"],
        check=True,
    )

print("Системная подготовка завершена.")


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

repo = Path("/content/audio-restoration-colab")
repo_url = "https://github.com/egor125552/audio-restoration-colab.git"
branch = "agent/stem-separator-mixer"

if not (repo / ".git").is_dir():
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "--branch", branch,
            repo_url, str(repo),
        ],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(repo), "fetch", "origin", branch, "--depth", "1"],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(repo), "checkout", "-B", branch, f"origin/{branch}"],
        check=True,
    )

uv = shutil.which("uv") or "/root/.local/bin/uv"
subprocess.run([uv, "python", "install", "3.11"], check=True)
subprocess.run(
    [
        uv, "venv", "--allow-existing", "--python", "3.11",
        str(repo / ".venv"),
    ],
    check=True,
)
subprocess.run(
    [
        uv, "pip", "install",
        "--python", str(repo / ".venv/bin/python"),
        str(repo),
    ],
    check=True,
)

os.chdir(repo)
print("Интерфейс установлен из ветки:", branch)


In [ ]:
import subprocess

print(
    "Ставлю среду разделителя: CUDA PyTorch, audio-separator "
    "и registry-backed BS-RoFormer…",
    flush=True,
)
subprocess.run(
    [
        "/content/audio-restoration-colab/scripts/prepare_backend.sh",
        "separator",
        "/content/audio-restoration-models",
    ],
    check=True,
)
print(
    "Среда разделителя готова. Веса конкретной модели "
    "скачаются один раз при первом выборе.",
    flush=True,
)


In [ ]:
import os
import re
import subprocess
import threading
import time
from pathlib import Path

repo = Path("/content/audio-restoration-colab")
log_path = Path("/content/audio-restoration-gradio.log")

old_process = globals().get("gradio_process")
if old_process is not None and old_process.poll() is None:
    old_process.terminate()
    old_process.wait(timeout=15)

log_path.write_text("", encoding="utf-8")
gradio_process = subprocess.Popen(
    [
        str(repo / ".venv/bin/python"),
        "-m",
        "audio_restoration_colab.app",
        "--share",
    ],
    cwd=repo,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env={
        **os.environ,
        "PYTHONUNBUFFERED": "1",
        "AUDIO_RESTORATION_CACHE": "/content/audio-restoration-models",
    },
)

def relay_output(process, path):
    with path.open("a", encoding="utf-8") as log_file:
        if process.stdout is None:
            return
        for line in process.stdout:
            print(line, end="", flush=True)
            log_file.write(line)
            log_file.flush()

threading.Thread(
    target=relay_output,
    args=(gradio_process, log_path),
    daemon=True,
).start()

print("Запускаю интерфейс…", flush=True)
for _ in range(180):
    time.sleep(1)
    log_text = log_path.read_text(
        encoding="utf-8",
        errors="replace",
    )
    match = re.search(r"https://[^\s]+\.gradio\.live", log_text)
    if match:
        print("Интерфейс готов:", match.group(0))
        print("Лог:", log_path)
        break
    if gradio_process.poll() is not None:
        raise RuntimeError(
            "Gradio завершился с ошибкой:\n" + log_text[-4000:]
        )
else:
    gradio_process.terminate()
    raise RuntimeError(
        "Gradio не выдал ссылку за 3 минуты:\n" + log_text[-4000:]
    )
